# Advanced Data Reshaping & Transformation with Pandas


In data science, real-world data rarely arrives in the perfect shape needed for machine learning models, statistical analysis, or visualization. Instead of a tidy grid, data often looks like a tangled knot.

Today, we will master **four essential advanced structural transformations** in Pandas that allow you to reshape data like clay:

1. **Multi-Indexing** (Hierarchical Data Organization)
2. **Melting** (`pd.melt()`) (Wide to Long transformation)
3. **Pivoting** (`pd.pivot()` and `pd.pivot_table()`) (Long to Wide transformation)
4. **Binning** (`pd.cut()`) (Continuous to Categorical conversion)

---

## 1. Multi-Indexing (Hierarchical Data)

### The Intuition & Analogy

Imagine a massive digital file cabinet for a global retail chain. If you organize files by just "City", you will have a hard time if multiple countries have a city named "Springfield." If you organize solely by "Year," files from different stores get mixed up.

A **MultiIndex** (or hierarchical index) is like a multi-level filing system: **[Country $\rightarrow$ City $\rightarrow$ Store ID]**. It allows you to store and work with higher-dimensional data (3D, 4D, or more) within a standard 2D spreadsheet or DataFrame structure without flattening or repeating values unnecessarily.

### The "Why"

Instead of creating clunky, concatenated string keys (like `"US_Springfield_Store1"`), multi-indexing allows Pandas to group data natively. It optimizes memory, speeds up search operations, and enables effortless multi-level aggregations (e.g., calculating sales at the city level vs. the country level in a single line).

### Progressive Code Example

Let's build a MultiIndex DataFrame representing regional sales over two years.

In [ ]:
import pandas as pd
import numpy as np

# Step 1: Defining the multi-level categories
regions = ['North', 'North', 'South', 'South']
cities = ['Chicago', 'Detroit', 'Houston', 'Miami']

# Create a MultiIndex from arrays
index = pd.MultiIndex.from_arrays([regions, cities], names=['Region', 'City'])

# Step 2: Build the DataFrame
data = {
    '2024_Sales': [150000, 95000, 120000, 185000],
    '2025_Sales': [165000, 98000, 130000, 190000]
}
sales_df = pd.DataFrame(data, index=index)
print("--- Hierarchical DataFrame ---")
print(sales_df)

--- Hierarchical DataFrame ---
                2024_Sales  2025_Sales
Region City                           
North  Chicago      150000      165000
       Detroit       95000       98000
South  Houston      120000      130000
       Miami        185000      190000


In [ ]:
sales_df

# "_".join()

# temp.apply(lambda x : (x["Region"] + "_" + x["City"]), axis=1)

2024_Sales  2025_Sales
Region City                           
North  Chicago      150000      165000
       Detroit       95000       98000
South  Houston      120000      130000
       Miami        185000      190000

**Output:**

```text
--- Hierarchical DataFrame ---
                2024_Sales  2025_Sales
Region City                           
North  Chicago      150000      165000
       Detroit       95000       98000
South  Houston      120000      130000
       Miami        185000      190000

```

### Navigating the MultiIndex (Advanced Depth)

How do we extract data out of this multi-tiered architecture? We use `.loc` alongside a Python `tuple`

In [ ]:
# Accessing an entire top-level group
print(sales_df.loc['North'])

# Accessing a specific nested row (Region='North', City='Chicago')

         2024_Sales  2025_Sales
City                           
Chicago      150000      165000
Detroit       95000       98000


In [ ]:
print(sales_df.loc[('North', 'Chicago')])


2024_Sales    150000
2025_Sales    165000
Name: (North, Chicago), dtype: int64


### Reflection & Quick Task

**Task:** Given the `sales_df`, use the `.groupby(level='Region').sum()` or `.groupby(level=0).sum()` method to see how effortlessly Pandas aggregates multi-indexed data across an entire structural tier.
*Question for thought:* When would you prefer a MultiIndex over simply keeping `Region` and `City` as regular columns? (Hint: Think about join operations and matrix representations).

---

## 2. Melting with `pd.melt()`

### The Intuition & Analogy

Imagine you have a spreadsheet tracking student test scores. The columns are: `[Student_Name, Math_Score, Physics_Score, Chemistry_Score]`.

This is a **Wide Format**. It’s highly readable for humans. However, if you want to create a general bar chart showing *all* scores across subjects, or feed this data into a machine learning model forecasting "Score" based on "Subject", Python wants a **Long Format** layout: `[Student_Name, Subject, Score]`.

`pd.melt()` acts like a physical candle melt. It takes separate, rigid columns (`Math_Score`, `Physics_Score`) and unpivots ("melts") them down into rows, turning column names into variable values.

### The "Why"

Tidy Data principles (pioneered by Hadley Wickham) state that **each variable forms a column, and each observation forms a row**. Wide data violates this because the variable "Subject" is scattered across multiple column headers. Melting aligns your dataset with the Tidy format, which is the required input format for libraries like `seaborn` and most statistical modeling algorithms.

### Progressive Code Example

In [ ]:
# Starting with a "Wide" dataset
wide_df = pd.DataFrame({
    'Student': ['Alice', 'Bob'],
    'Math': [95, 82],
    'Physics': [89, 90],
    'Chemistry': [92, 78]
})
print("--- Wide Format ---")
print(wide_df)

# Melting the dataset down
long_df = pd.melt(
    wide_df, # Wide Format Dataframe
    id_vars=['Student'],                      # Columns to keep unchanged (identifiers)
    value_vars=['Math', 'Physics', 'Chemistry'], # Columns to unpivot/melt down
    var_name='Subject',                       # Rename the new identifier column
    value_name='Score'                        # Rename the new data value column
)
print("\n--- Long Format (Melted) ---")
print(long_df)

--- Wide Format ---
  Student  Math  Physics  Chemistry
0   Alice    95       89         92
1     Bob    82       90         78

--- Long Format (Melted) ---
  Student    Subject  Score
0   Alice       Math     95
1     Bob       Math     82
2   Alice    Physics     89
3     Bob    Physics     90
4   Alice  Chemistry     92
5     Bob  Chemistry     78


**Output:**

```text
--- Wide Format ---
  Student  Math  Physics  Chemistry
0   Alice    95       89         92
1     Bob    82       90         78

--- Long Format (Melted) ---
  Student    Subject  Score
0   Alice       Math     95
1     Bob       Math     82
2   Alice    Physics     89
3     Bob    Physics     90
4   Alice  Chemistry     92
5     Bob  Chemistry     78

```

---

## 3. Pivoting (`pd.pivot()` vs `pd.pivot_table()`)

### The Intuition & Analogy

Pivoting is the exact mathematical inverse of melting. It takes a **Long Format** dataset and stretches it horizontally into a **Wide Format** summary matrix.

* `pd.pivot()` is an organizer. Use it when your long data has unique row-column combinations and just needs to be rearranged structurally.
* `pd.pivot_table()` is a manager and an analytical powerhouse. Use it when you have duplicate entries for a row-column combination and need to **aggregate** them (e.g., averaging, summing) while restructuring.

### The "Why"

Machine learning models (like Scikit-Learn algorithms) expect a feature matrix where each column represents an independent attribute. If your web traffic logs show one event per row, you must pivot the data so that each unique visitor gets exactly *one* row, and their page-views per category become distinct columns.

### Progressive Code Example: `pd.pivot()`

In [ ]:
# Long data with unique observations per combination
log_df = pd.DataFrame({
    'Date': ['2026-05-01', '2026-05-01', '2026-05-02', '2026-05-02'],
    'Device': ['Mobile', 'Desktop', 'Mobile', 'Desktop'],
    'Clicks': [120, 240, 135, 210]
})

# Reshape without aggregation
pivoted_df = log_df.pivot(index='Date', columns='Device', values='Clicks')
print("--- pd.pivot() Matrix ---")
print(pivoted_df)

--- pd.pivot() Matrix ---
Device      Desktop  Mobile
Date                       
2026-05-01      240     120
2026-05-02      210     135


In [ ]:
log_df

,Date,Device,Clicks
0,2026-05-01,Mobile,120
1,2026-05-01,Desktop,240
2,2026-05-02,Mobile,135
3,2026-05-02,Desktop,210


**Output:**

```text
--- pd.pivot() Matrix ---
Device      Desktop  Mobile
Date                       
2026-05-01      240     120
2026-05-02      210     135

```

### Advanced Depth: `pd.pivot_table()` with Duplicates

What happens if our log file has multiple entries for `Mobile` on the same exact day? `pd.pivot()` will crash and throw a `ValueError: Index contains duplicate entries`. Enter `pd.pivot_table()`.

In [ ]:
pivoted_df = log_df.pivot(index='Date', columns='Device', values='Clicks')

In [ ]:
# Raw transaction data (Notice multiple 'Mobile' entries on Day 1)
raw_sales = pd.DataFrame({
    'Date': ['05-01', '05-01', '05-01', '05-02'],
    'Device': ['Mobile', 'Mobile', 'Desktop', 'Mobile'],
    'Revenue': [100, 150, 400, 300]
})

# pivot_table defaults to aggfunc='mean', but we can explicitly set 'sum'
pivot_summary = raw_sales.pivot_table(
    index='Date',
    columns='Device',
    values='Revenue',
    aggfunc='sum',
    fill_value=0 # Handles missing structural combinations gracefully
)
print("--- pd.pivot_table() Summary (Aggregated Sum) ---")
print(pivot_summary)

--- pd.pivot_table() Summary (Aggregated Sum) ---
Device  Desktop  Mobile
Date                   
05-01       400     250
05-02         0     300


**Output:**

```text
--- pd.pivot_table() Summary (Aggregated Sum) ---
Device  Desktop  Mobile
Date                   
05-01       400     250
05-02         0     300

```

### Reflection & Exercise

**Mini-Coding Challenge:** Take the `pivot_summary` DataFrame and apply `.melt()` to reconstruct the long-format data. Notice how easily you can cycle back and forth between these formats once you master the parameters!

---

## 4. Binning with `pd.cut()`

### The Intuition & Analogy

Imagine sorting thousands of physical letters into delivery bins based on their destination zip codes, or sorting apples by their size into "Small," "Medium," and "Large" boxes.

In data science, treating a continuous variable (like Age or Income) as an infinite gradient can sometimes muddy your analysis. **Binning** with `pd.cut()` draws strict mathematical lines in a continuous numerical spectrum, dividing it into discrete, easily digestible chunks or buckets.

### The "Why"

1. **Business Intelligence:** Executives don't typically look at data for 27.4-year-olds vs 27.5-year-olds; they analyze demographic brackets (e.g., "Gen Z", "Millennials").
2. **Algorithm Requirements:** Certain machine learning algorithms (like Naive Bayes or decision tree variants) handle discrete categorical factors more robustly than raw, skewed continuous features.

### Progressive Code Example

Let's convert raw customer credit scores into categorical risk tiers.

In [ ]:
# Sample credit scores of applicants
credit_scores = [520, 790, 640, 580, 810, 710, 620]

# Define structural boundary edges for our bins
# Bins: 500-600 (Poor), 601-700 (Fair), 701-800 (Good), 801-850 (Excellent)
bins = [500, 600, 700, 800, 850]
tier_labels = ['Poor', 'Fair', 'Good', 'Excellent']

# Segment the continuous data
score_tiers = pd.cut(credit_scores, bins=bins, labels=tier_labels)

# Put into a clear DataFrame
credit_df = pd.DataFrame({
    'Score': credit_scores,
    'Risk_Tier': score_tiers
})
print("--- Binned Continuous Data ---")
print(credit_df)

--- Binned Continuous Data ---
   Score  Risk_Tier
0    520       Poor
1    790       Good
2    640       Fair
3    580       Poor
4    810  Excellent
5    710       Good
6    620       Fair
